# Stage 09 / Lecture 65: Dataclasses and Unified Agent `response_format` Integration

This production reference notebook demonstrates integrating structured outputs directly into **`create_agent(..., response_format=Schema)`**, contrasting all three supported schema paradigms side-by-side:
1. **Pydantic `BaseModel`**: Strictly validated objects returned in `result["structured_response"]`.
2. **`TypedDict`**: Lightweight zero-validation dictionaries returned in `result["structured_response"]`.
3. **Python `@dataclass`**: Native zero-dependency attribute-accessible data containers (introduced in Python 3.7) returned in `result["structured_response"]`.


In [1]:
from typing import Dict, Any
from dataclasses import dataclass, is_dataclass
from typing_extensions import TypedDict
import os
from dotenv import load_dotenv

from pydantic import BaseModel, ValidationError
from langchain.agents import create_agent
from langchain_groq import ChatGroq

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY must be configured in .env for agent response_format verification.")

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
sample_query = {
    "messages": [
        {"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}
    ]
}
print("Model and extraction prompt initialized successfully.")


Model and extraction prompt initialized successfully.


### 1. Integrating `Pydantic` and `TypedDict` via `create_agent(..., response_format=...)`

Passing `response_format` to `create_agent()` appends a structured payload under `result["structured_response"]` alongside the agent's standard `result["messages"]` trace.


In [2]:
# Option A: Pydantic BaseModel (strict runtime validation)
class ContactPydantic(BaseModel):
    name: str
    email: str
    phone: str

agent_pydantic = create_agent(model=model, response_format=ContactPydantic)
res_pydantic = agent_pydantic.invoke(sample_query)
pydantic_out = res_pydantic["structured_response"]

assert isinstance(pydantic_out, ContactPydantic)
assert pydantic_out.name == "John Doe"
assert pydantic_out.email == "john@example.com"
assert "555" in pydantic_out.phone

# Option B: TypedDict (plain dict, zero validation)
class ContactTypedDict(TypedDict):
    name: str
    email: str
    phone: str

agent_typedict = create_agent(model=model, response_format=ContactTypedDict)
res_typedict = agent_typedict.invoke(sample_query)
typedict_out = res_typedict["structured_response"]

assert isinstance(typedict_out, dict)
assert typedict_out["name"] == "John Doe"
assert typedict_out["email"] == "john@example.com"

print("1. Agent with Pydantic response_format:", pydantic_out)
print("2. Agent with TypedDict response_format:", typedict_out)


1. Agent with Pydantic response_format: name='John Doe' email='john@example.com' phone='(555) 123-4567'
2. Agent with TypedDict response_format: {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


### 2. Integrating Python `@dataclass` inside `create_agent()`

`@dataclass` offers clean attribute access (`contact.name`, `contact.email`) using only Python's standard library, while omitting runtime type enforcement.


In [3]:
@dataclass
class ContactDataclass:
    name: str
    email: str
    phone: str


# Demonstrate lack of runtime type validation compared to Pydantic
unvalidated_dc = ContactDataclass(name=12345, email="john@example.com", phone=999)
assert is_dataclass(unvalidated_dc) and unvalidated_dc.name == 12345

try:
    ContactPydantic(name=12345, email="john@example.com", phone=999)
    raise AssertionError("Expected ValidationError for Pydantic.")
except ValidationError:
    print("Confirmed: @dataclass omits runtime type checks while Pydantic enforces them.")

# Execute Agent with @dataclass response_format
agent_dataclass = create_agent(model=model, response_format=ContactDataclass)
res_dataclass: Dict[str, Any] = agent_dataclass.invoke(sample_query)

assert "messages" in res_dataclass and "structured_response" in res_dataclass
dc_out: ContactDataclass = res_dataclass["structured_response"]

assert isinstance(dc_out, ContactDataclass)
assert dc_out.name == "John Doe"
assert dc_out.email == "john@example.com"
assert "555" in dc_out.phone

print(f"3. Agent with @dataclass response_format: {dc_out}")
print(f"   Dot-access check -> Name: {dc_out.name} | Email: {dc_out.email} | Phone: {dc_out.phone}")
print("\nAll Stage 09 / Lecture 65 dataclass and create_agent structured_response assertions passed.")


Confirmed: @dataclass omits runtime type checks while Pydantic enforces them.


3. Agent with @dataclass response_format: ContactDataclass(name='John Doe', email='john@example.com', phone='(555) 123-4567')
   Dot-access check -> Name: John Doe | Email: john@example.com | Phone: (555) 123-4567

All Stage 09 / Lecture 65 dataclass and create_agent structured_response assertions passed.
